# Проверка X_INN по трем выгрузкам

Тетрадка загружает 3 файла, применяет фильтрацию по методике `analysis_crm_segments.ipynb`, считает уникальные `X_INN` и проверяет пересечения множеств ИНН.

In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

In [ ]:
# Пути к данным (по вашей договоренности)
DATA_DIR = Path("/home/jovyan/documents/DMUKP_FP_DEF/data")

FILES = {
    "crm_last_version": DATA_DIR / "crm_last_version.csv",
    "crm_all_clients": DATA_DIR / "Выгрузка_CRM_все_клиенты.csv",
    "fp_by_inn": DATA_DIR / "Выгрузка_ФП_по_ИНН.xlsx",
}

for name, path in FILES.items():
    print(f"{name}: {path}")

In [ ]:
DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")
ALLOWED_SOURCES = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    if len(s) <= 10:
        return s.zfill(10)
    return s.zfill(12)


def prepare_df(raw_df: pd.DataFrame, file_label: str) -> tuple[pd.DataFrame, dict]:
    """Подготовка данных с разной логикой для разных источников."""
    df = raw_df.copy()
    stats = {"file": file_label, "rows_raw": len(df)}

    if "X_INN" not in df.columns:
        raise KeyError(f"В файле {file_label} нет колонки X_INN")

    # Базовая подготовка для всех файлов
    df["inn"] = df["X_INN"].apply(normalize_inn)

    if "X_AREA_RESP" in df.columns:
        df["X_AREA_RESP"] = df["X_AREA_RESP"].astype(str).str.strip()
        df["segment"] = df["X_AREA_RESP"].map(SEGMENT_MAP)

    # Для all_clients: берем только разрешенные сегменты (как в analysis_crm_segments),
    # затем оставляем уникальные inn.
    if file_label == "crm_all_clients":
        if "X_AREA_RESP" in df.columns:
            df = df[df["segment"].notna()].copy()
        else:
            raise KeyError("В файле crm_all_clients нет колонки X_AREA_RESP для фильтра сегментов")

        df = df.dropna(subset=["inn"]).copy()
        df = df.drop_duplicates(subset=["inn"]).copy()

    else:
        # Логика analysis_crm_segments для CRM/FP источников
        if "IDENTIFICATION_DATE" in df.columns:
            df["dt"] = pd.to_datetime(df["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
            df = df[(df["dt"] >= DATE_FROM) & (df["dt"] <= DATE_TO)].copy()

        if "VAL" in df.columns:
            df = df[df["VAL"].astype(str).str.strip().isin(ALLOWED_SOURCES)].copy()

        if "TYPE_FP" in df.columns:
            df["TYPE_FP"] = df["TYPE_FP"].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})

        if "X_AREA_RESP" in df.columns:
            df = df[df["segment"].notna()].copy()

        if "NUMBER_FP_SFP" in df.columns:
            df["fp_num"] = df["NUMBER_FP_SFP"].astype(str).str.strip()
            df = df.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()

        dedup_cols = [c for c in ["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"] if c in df.columns]
        if dedup_cols:
            df = df.drop_duplicates(subset=dedup_cols).copy()
        else:
            df = df.drop_duplicates(subset=["inn"]).copy()

    stats["rows_filtered"] = len(df)
    stats["unique_x_inn"] = df["X_INN"].astype(str).nunique(dropna=True)
    stats["unique_inn_norm"] = df["inn"].nunique(dropna=True)

    return df, stats

In [ ]:
def read_semicolon_keep_all(path: Path, skip_rows: int = 33, sep: str = ";") -> pd.DataFrame:
    """Читает CSV после skip_rows без потери строк (по аналогии с analysis_crm_all_clients)."""
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    if len(lines) <= skip_rows:
        raise ValueError(f"Недостаточно строк после skip_rows={skip_rows}: {path}")

    header = lines[skip_rows].rstrip("\n")
    columns = header.split(sep)
    ncols = len(columns)

    comment_candidates = ["comment", "COMMENT", "COMMENT_TEXT"]
    comment_idx = next((columns.index(c) for c in comment_candidates if c in columns), None)

    records = []
    fixed_more = 0
    fixed_less = 0

    for raw_line in lines[skip_rows + 1 :]:
        raw = raw_line.rstrip("\n")
        parts = raw.split(sep)

        if len(parts) == ncols:
            records.append(parts)
            continue

        if len(parts) > ncols and comment_idx is not None:
            extra = len(parts) - ncols
            merged_comment = sep.join(parts[comment_idx : comment_idx + extra + 1])
            repaired = parts[:comment_idx] + [merged_comment] + parts[comment_idx + extra + 1 :]
            if len(repaired) == ncols:
                records.append(repaired)
                fixed_more += 1
                continue

        if len(parts) < ncols:
            records.append(parts + [""] * (ncols - len(parts)))
            fixed_less += 1
            continue

        tail = sep.join(parts[ncols - 1 :])
        repaired = parts[: ncols - 1] + [tail]
        records.append(repaired)
        fixed_more += 1

    df_out = pd.DataFrame(records, columns=columns)
    print(
        f"{path.name}: rows={len(df_out):,}, cols={df_out.shape[1]}, "
        f"fixed_more={fixed_more:,}, fixed_less={fixed_less:,}"
    )
    return df_out


def load_file(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Файл не найден: {path}")

    if path.suffix.lower() == ".csv":
        # Для all_clients используем спец-чтение из analysis_crm_all_clients.ipynb
        if path.name == "Выгрузка_CRM_все_клиенты.csv":
            return read_semicolon_keep_all(path=path, skip_rows=33, sep=";")

        # Базовый вариант из методики segments
        return pd.read_csv(path, dtype=str, low_memory=False)

    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, dtype=str)

    raise ValueError(f"Неподдерживаемый формат: {path}")


raw_data = {}
prepared_data = {}
stats_rows = []

for file_key, file_path in FILES.items():
    raw_df = load_file(file_path)
    df_prepared, stats = prepare_df(raw_df, file_key)

    raw_data[file_key] = raw_df
    prepared_data[file_key] = df_prepared
    stats_rows.append(stats)

    print(f"{file_key}: raw={stats['rows_raw']:,}, filtered={stats['rows_filtered']:,}, unique_inn={stats['unique_inn_norm']:,}")

In [ ]:
stats_df = pd.DataFrame(stats_rows)
stats_df = stats_df[["file", "rows_raw", "rows_filtered", "unique_x_inn", "unique_inn_norm"]]
stats_df = stats_df.sort_values("file").reset_index(drop=True)

print("Статистика по файлам:")
display(stats_df)

In [ ]:
sets = {k: set(v["inn"].dropna().unique()) for k, v in prepared_data.items()}

A = sets["crm_last_version"]
B = sets["crm_all_clients"]
C = sets["fp_by_inn"]

overlap_stats = pd.DataFrame([
    {"metric": "|A| crm_last_version", "value": len(A)},
    {"metric": "|B| crm_all_clients", "value": len(B)},
    {"metric": "|C| fp_by_inn", "value": len(C)},
    {"metric": "|A ∩ B|", "value": len(A & B)},
    {"metric": "|A ∩ C|", "value": len(A & C)},
    {"metric": "|B ∩ C|", "value": len(B & C)},
    {"metric": "|A ∩ B ∩ C|", "value": len(A & B & C)},
    {"metric": "|A \\ B|", "value": len(A - B)},
    {"metric": "|A \\ C|", "value": len(A - C)},
    {"metric": "|B \\ A|", "value": len(B - A)},
    {"metric": "|B \\ C|", "value": len(B - C)},
    {"metric": "|C \\ A|", "value": len(C - A)},
    {"metric": "|C \\ B|", "value": len(C - B)},
])

expected_c = A - B
check_expected = pd.DataFrame([
    {"metric": "|expected_c = A \\ B|", "value": len(expected_c)},
    {"metric": "|C|", "value": len(C)},
    {"metric": "C == expected_c", "value": C == expected_c},
    {"metric": "|C \\ expected_c|", "value": len(C - expected_c)},
    {"metric": "|expected_c \\ C|", "value": len(expected_c - C)},
])

print("Пересечения и разности:")
display(overlap_stats)
print("Проверка, что fp_by_inn == crm_last_version - crm_all_clients:")
display(check_expected)

In [ ]:
# Показываем примеры расхождений (если есть)
examples = {
    "C_minus_expected": sorted(list(C - expected_c))[:50],
    "expected_minus_C": sorted(list(expected_c - C))[:50],
    "A_minus_B": sorted(list(A - B))[:50],
}

for key, values in examples.items():
    print(f"\n{key}: {len(values)} примеров")
    print(values[:10])

samples_df = pd.DataFrame({
    "C_minus_expected": pd.Series(examples["C_minus_expected"]),
    "expected_minus_C": pd.Series(examples["expected_minus_C"]),
    "A_minus_B": pd.Series(examples["A_minus_B"]),
})

display(samples_df.head(50))

In [ ]:
# Опциональный экспорт результатов
EXPORT_RESULTS = False
EXPORT_PATH = Path("e:/DMUKP/final_report/crm_inn_crosscheck_results.xlsx")

if EXPORT_RESULTS:
    with pd.ExcelWriter(EXPORT_PATH, engine="openpyxl") as writer:
        stats_df.to_excel(writer, sheet_name="inn_stats", index=False)
        overlap_stats.to_excel(writer, sheet_name="set_overlaps", index=False)
        check_expected.to_excel(writer, sheet_name="expected_check", index=False)
        samples_df.to_excel(writer, sheet_name="sample_diffs", index=False)
        if "all_clients_seg" in globals():
            all_clients_seg.to_excel(writer, sheet_name="seg_all_clients", index=False)
        if "missing_seg" in globals():
            missing_seg.to_excel(writer, sheet_name="seg_missing_from_last", index=False)
        if "segment_total" in globals():
            segment_total.to_excel(writer, sheet_name="seg_total_with_missing", index=False)
        if "crm_last_version_seg" in globals():
            crm_last_version_seg.to_excel(writer, sheet_name="seg_crm_last_version", index=False)
        if "compare_segments" in globals():
            compare_segments.to_excel(writer, sheet_name="seg_compare", index=False)

        # Новые листы по ГСЗ/ГСК/ГК
        if "total_inn_df" in globals():
            total_inn_df.to_excel(writer, sheet_name="inn_union_with_names", index=False)
        if "coverage_summary" in globals():
            coverage_summary.to_excel(writer, sheet_name="name_coverage_summary", index=False)
        if "coverage_by_dataset" in globals():
            coverage_by_dataset.to_excel(writer, sheet_name="name_coverage_by_dataset", index=False)

        if "group_tables" in globals():
            if "NAME_1" in group_tables:
                group_tables["NAME_1"].to_excel(writer, sheet_name="gsz_name1", index=False)
            if "NAME_2" in group_tables:
                group_tables["NAME_2"].to_excel(writer, sheet_name="gsk_name2", index=False)
            if "NAME_3" in group_tables:
                group_tables["NAME_3"].to_excel(writer, sheet_name="gk_name3", index=False)

        if "fill_tables" in globals():
            if "NAME_1" in fill_tables:
                fill_tables["NAME_1"].to_excel(writer, sheet_name="fill_name1", index=False)
            if "NAME_2" in fill_tables:
                fill_tables["NAME_2"].to_excel(writer, sheet_name="fill_name2", index=False)
            if "NAME_3" in fill_tables:
                fill_tables["NAME_3"].to_excel(writer, sheet_name="fill_name3", index=False)

        if "missing_without_any_group" in globals():
            missing_without_any_group.to_excel(writer, sheet_name="missing_no_groups", index=False)

    print(f"Результаты сохранены: {EXPORT_PATH}")
else:
    print("Экспорт выключен. Установите EXPORT_RESULTS = True, чтобы сохранить файл.")

In [ ]:
# Разбивка по сегментам: база crm_all_clients + добавка отсутствующих ИНН из crm_last_version

# 1) Уникальные ИНН в crm_all_clients по segment
all_clients_seg = (
    prepared_data["crm_all_clients"]
    .dropna(subset=["inn", "segment"])
    .groupby("segment", as_index=False)["inn"]
    .nunique()
    .rename(columns={"inn": "unique_inn"})
)

# 2) ИНН, которых нет в crm_all_clients, но есть в crm_last_version
missing_in_all_clients = A - B
missing_df = prepared_data["crm_last_version"][prepared_data["crm_last_version"]["inn"].isin(missing_in_all_clients)].copy()

missing_seg = (
    missing_df
    .dropna(subset=["inn", "segment"])
    .groupby("segment", as_index=False)["inn"]
    .nunique()
    .rename(columns={"inn": "missing_inn_from_crm_last_version"})
)

# 3) Итоговая разбивка: all_clients + добавка из crm_last_version
segment_total = all_clients_seg.merge(missing_seg, on="segment", how="outer").fillna(0)
segment_total["unique_inn"] = segment_total["unique_inn"].astype(int)
segment_total["missing_inn_from_crm_last_version"] = segment_total["missing_inn_from_crm_last_version"].astype(int)
segment_total["total_with_missing"] = (
    segment_total["unique_inn"] + segment_total["missing_inn_from_crm_last_version"]
)
segment_total = segment_total.sort_values("total_with_missing", ascending=False).reset_index(drop=True)

print("Разбивка уникальных ИНН по сегментам (crm_all_clients):")
display(all_clients_seg)

print(f"ИНН в crm_last_version, отсутствующих в crm_all_clients: {len(missing_in_all_clients):,}")
print("Разбивка этих ИНН по сегментам:")
display(missing_seg)

print("Итог (crm_all_clients + добавка отсутствующих из crm_last_version):")
display(segment_total)

In [ ]:
# Отдельная разбивка по сегментам из crm_last_version и сравнение с итоговой сборкой

crm_last_version_seg = (
    prepared_data["crm_last_version"]
    .dropna(subset=["inn", "segment"])
    .groupby("segment", as_index=False)["inn"]
    .nunique()
    .rename(columns={"inn": "crm_last_version_unique_inn"})
)

compare_segments = (
    crm_last_version_seg
    .merge(
        segment_total[["segment", "total_with_missing"]],
        on="segment",
        how="outer",
    )
    .fillna(0)
)

compare_segments["crm_last_version_unique_inn"] = compare_segments["crm_last_version_unique_inn"].astype(int)
compare_segments["total_with_missing"] = compare_segments["total_with_missing"].astype(int)
compare_segments["diff_total_minus_last"] = (
    compare_segments["total_with_missing"] - compare_segments["crm_last_version_unique_inn"]
)
compare_segments = compare_segments.sort_values("segment").reset_index(drop=True)

print("Разбивка уникальных ИНН по сегментам (crm_last_version):")
display(crm_last_version_seg)

print("Сравнение сегментов: (crm_all_clients + missing из crm_last_version) vs crm_last_version")
display(compare_segments)

if (compare_segments["diff_total_minus_last"] == 0).all():
    print("Расхождений по сегментам нет: значения совпадают.")
else:
    print("Есть расхождения по сегментам. Смотрите столбец diff_total_minus_last.")

In [ ]:
# Единая витрина ИНН и подтяжка ГСЗ/ГСК/ГК (NAME_1/NAME_2/NAME_3)

NAME_COLS = ["NAME_1", "NAME_2", "NAME_3"]
MISSING_MARKERS = {"", "nan", "None", "<NA>"}


def clean_name_columns(df_src: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df_src.copy()
    for col in cols:
        if col not in df.columns:
            df[col] = pd.NA
        df[col] = df[col].astype(str).str.strip()
        df.loc[df[col].isin(MISSING_MARKERS), col] = pd.NA
    return df


def first_notna(series: pd.Series):
    non_na = series.dropna()
    return non_na.iloc[0] if len(non_na) else pd.NA


# 1) База и расширенная популяция ИНН
inn_all_clients = set(prepared_data["crm_all_clients"]["inn"].dropna().astype(str).unique())
inn_last_version = set(prepared_data["crm_last_version"]["inn"].dropna().astype(str).unique())
missing_from_all_clients = inn_last_version - inn_all_clients

base_inn_df = pd.DataFrame({"inn": sorted(inn_all_clients)})
base_inn_df["source"] = "crm_all_clients"

missing_inn_df = pd.DataFrame({"inn": sorted(missing_from_all_clients)})
missing_inn_df["source"] = "crm_last_version_only"

total_inn_df = pd.concat([base_inn_df, missing_inn_df], ignore_index=True)

# 2) Основной lookup NAME_* из crm_all_clients
all_clients_names = clean_name_columns(prepared_data["crm_all_clients"][["inn", *NAME_COLS]], NAME_COLS)
lookup_all_clients = (
    all_clients_names.groupby("inn", as_index=False)
    .agg({col: first_notna for col in NAME_COLS})
)

# 3) Fallback lookup NAME_* из fp_by_inn
fp_names = clean_name_columns(prepared_data["fp_by_inn"][["inn", *NAME_COLS]], NAME_COLS)
lookup_fp = (
    fp_names.groupby("inn", as_index=False)
    .agg({col: first_notna for col in NAME_COLS})
)

# 4) Итоговая витрина с приоритетом crm_all_clients, затем fp_by_inn
total_inn_df = total_inn_df.merge(lookup_all_clients, on="inn", how="left")
total_inn_df = total_inn_df.merge(lookup_fp, on="inn", how="left", suffixes=("", "_fp"))

for col in NAME_COLS:
    total_inn_df[col] = total_inn_df[col].fillna(total_inn_df[f"{col}_fp"])

drop_cols = [f"{col}_fp" for col in NAME_COLS]
total_inn_df = total_inn_df.drop(columns=drop_cols)

# Флаги заполненности
for col in NAME_COLS:
    total_inn_df[f"has_{col}"] = total_inn_df[col].notna()

total_inn_df["has_any_group"] = total_inn_df[[f"has_{c}" for c in NAME_COLS]].any(axis=1)

# 5) Метрики покрытия
coverage_summary = pd.DataFrame([
    {"metric": "Уникальных ИНН в crm_all_clients", "value": len(inn_all_clients)},
    {"metric": "Уникальных ИНН в crm_last_version", "value": len(inn_last_version)},
    {"metric": "ИНН только из crm_last_version (вне crm_all_clients)", "value": len(missing_from_all_clients)},
    {"metric": "Уникальных ИНН в расширенной базе", "value": len(total_inn_df)},
    {"metric": "С заполненным NAME_1 (ГСЗ)", "value": int(total_inn_df["has_NAME_1"].sum())},
    {"metric": "С заполненным NAME_2 (ГСК)", "value": int(total_inn_df["has_NAME_2"].sum())},
    {"metric": "С заполненным NAME_3 (ГК)", "value": int(total_inn_df["has_NAME_3"].sum())},
    {"metric": "С хотя бы одной группой (NAME_1/2/3)", "value": int(total_inn_df["has_any_group"].sum())},
])

crm_last_on_total = total_inn_df[total_inn_df["inn"].isin(inn_last_version)].copy()
crm_all_on_total = total_inn_df[total_inn_df["inn"].isin(inn_all_clients)].copy()

coverage_by_dataset = pd.DataFrame([
    {
        "dataset": "crm_all_clients",
        "unique_inn": len(crm_all_on_total),
        "has_any_group": int(crm_all_on_total["has_any_group"].sum()),
    },
    {
        "dataset": "crm_last_version",
        "unique_inn": len(crm_last_on_total),
        "has_any_group": int(crm_last_on_total["has_any_group"].sum()),
    },
])
coverage_by_dataset["share_has_any_group_pct"] = (
    coverage_by_dataset["has_any_group"] / coverage_by_dataset["unique_inn"] * 100
).round(2)

print("Свод по покрытию группами:")
display(coverage_summary)
print("Покрытие хотя бы одной группой по наборам:")
display(coverage_by_dataset)
print("Пример итоговой витрины ИНН:")
display(total_inn_df.head(20))

In [ ]:
# Разбивки по ГСЗ/ГСК/ГК (по уникальным ИНН итоговой витрины)

group_tables = {}
fill_tables = {}
label_map = {
    "NAME_1": "ГСЗ",
    "NAME_2": "ГСК",
    "NAME_3": "ГК",
}

for col in NAME_COLS:
    title = label_map[col]

    base = total_inn_df[["inn", col]].copy()
    base[col] = base[col].astype(str).str.strip()
    base.loc[base[col].isin(MISSING_MARKERS), col] = "[ПУСТО]"

    tbl = (
        base.groupby(col)["inn"]
        .nunique()
        .rename("Уникальных ИНН")
        .reset_index()
        .sort_values("Уникальных ИНН", ascending=False)
    )
    group_tables[col] = tbl

    filled_mask = base[col] != "[ПУСТО]"
    fill_tbl = pd.DataFrame([
        {"Статус": "Заполнено", "Уникальных ИНН": int(base.loc[filled_mask, "inn"].nunique())},
        {"Статус": "Пусто", "Уникальных ИНН": int(base.loc[~filled_mask, "inn"].nunique())},
    ])
    fill_tables[col] = fill_tbl

    print(f"Разбивка по {title} ({col}):")
    display(tbl)
    print(f"Заполненность {title} ({col}):")
    display(fill_tbl)

# Контроль: ИНН без групп среди отсутствующих в crm_all_clients
missing_only = total_inn_df[total_inn_df["source"] == "crm_last_version_only"].copy()
missing_without_any_group = missing_only[~missing_only["has_any_group"]]

print(f"\nИНН только из crm_last_version: {len(missing_only):,}")
print(f"Из них без ГСЗ/ГСК/ГК (пусто): {len(missing_without_any_group):,}")
display(missing_without_any_group[["inn", "NAME_1", "NAME_2", "NAME_3"]].head(50))